Author: Jessica Xu

In the first section, the code constructs a Chicago ZIP code CSV that is later used to scrape restaurants from Yelp. In the second and third sections, it cleans the Google Places and Yelp datasets via dropping columns with excessive missingness, extracting the key variables needed for analysis, and reorganizing the remaining columns.

AI used only for debugging and minor modifications.

In [1]:
import os
import numpy as np
import pandas as pd
import re

## Build a Chicago zipcode CSV 

In [2]:
# load dataset
BASE_DIR = os.getcwd()

GP_FILENAME = "Google_Place_Chicago_restaurants/gp_chicago.csv"
GP_FILEPATH = os.path.join(BASE_DIR, GP_FILENAME)

gp = pd.read_csv(GP_FILEPATH)
print(gp.columns)

Index(['gp_place_id', 'gp_name', 'gp_address', 'gp_latitude', 'gp_longitude',
       'gp_price_range_raw', 'gp_price_level', 'gp_rating', 'gp_review_count',
       'gp_types', 'gp_primary_type', 'gp_business_status',
       'gp_regular_opening_hours_raw', 'gp_takeout', 'gp_delivery',
       'gp_dine_in', 'gp_reservable', 'gp_query_lat', 'gp_query_lon'],
      dtype='object')


In [3]:
# make sure gp_address is string
gp["gp_address"] = gp["gp_address"].astype(str)

# extract address components using regex
pattern = r'^(.*?),\s*(.*?),\s*[A-Z]{2}\s*(\d{5}),\s*(.*)$'
gp[["gp_specific_address", "gp_city", "gp_zipcode", "gp_country"]] = gp["gp_address"].str.extract(pattern)

# check results
print(gp[["gp_address", "gp_specific_address", "gp_city", "gp_zipcode", "gp_country"]].head())

                                      gp_address    gp_specific_address  \
0   8311 S Baltimore Ave, Chicago, IL 60617, USA   8311 S Baltimore Ave   
1    7902 S Exchange Ave, Chicago, IL 60617, USA    7902 S Exchange Ave   
2         2911 E 79th St, Chicago, IL 60649, USA         2911 E 79th St   
3  8548 S Commercial Ave, Chicago, IL 60617, USA  8548 S Commercial Ave   
4  8465 S Commercial Ave, Chicago, IL 60617, USA  8465 S Commercial Ave   

   gp_city gp_zipcode gp_country  
0  Chicago      60617        USA  
1  Chicago      60617        USA  
2  Chicago      60649        USA  
3  Chicago      60617        USA  
4  Chicago      60617        USA  


In [4]:
unique_zip_count = gp["gp_zipcode"].nunique()
print(unique_zip_count)

65


In [5]:
#unique_zips = sorted(gp["gp_zipcode"].dropna().unique())
#for z in unique_zips:
  #  print(z)

In [6]:
# save unique zipcode as a new CSV; used to collect restaurants by zipcode with Yelp API 
# unique_zip_df = pd.DataFrame(unique_zips, columns=["gp_zipcode"])
#unique_zip_df.to_csv("chicago_unique_zipcodes.csv", index=False) 

In [7]:
# save cleaned gp_chicago as a new CSV
# gp.to_csv("gp_chicago_clean.csv", index=False)

## Clean up gp_chicago

In [8]:
# helper function used to keep zipcode as string (5 dights)
def to_zip_str(s):
    return s.astype(str).str.extract(r"(\d{5})", expand=False)

In [9]:
gp["gp_zipcode"] = to_zip_str(gp["gp_zipcode"])

In [10]:
# Standardize types
gp["gp_latitude"] = pd.to_numeric(gp["gp_latitude"], errors="coerce")
gp["gp_longitude"] = pd.to_numeric(gp["gp_longitude"], errors="coerce")
gp["gp_rating"] = pd.to_numeric(gp["gp_rating"], errors="coerce")
gp["gp_review_count"] = pd.to_numeric(gp["gp_review_count"], errors="coerce").fillna(0).astype(int)


In [11]:
# fix rows where zipcode is error
gp["gp_zipcode"] = gp["gp_zipcode"].astype(str)
gp["gp_zipcode"] = gp["gp_zipcode"].where(gp["gp_zipcode"].str.fullmatch(r"\d{5}"), np.nan)
gp.loc[gp["gp_zipcode"].isna(), "gp_zipcode"] = gp.loc[gp["gp_zipcode"].isna(), "gp_address"].astype(str).str.extract(r"(\d{5})", expand=False)
gp["gp_zipcode"] = gp["gp_zipcode"].astype(str)

In [12]:
gp.isnull().sum().sort_values(ascending=False)

gp_reservable                   1242
gp_price_range_raw               787
gp_price_level                   768
gp_dine_in                       494
gp_delivery                      477
gp_regular_opening_hours_raw     267
gp_takeout                       215
gp_rating                        122
gp_country                        27
gp_city                           27
gp_specific_address               27
gp_primary_type                    4
gp_zipcode                         0
gp_query_lon                       0
gp_query_lat                       0
gp_place_id                        0
gp_name                            0
gp_types                           0
gp_review_count                    0
gp_longitude                       0
gp_latitude                        0
gp_address                         0
gp_business_status                 0
dtype: int64

In [13]:
gp["gp_business_status"] == "OPERATIONAL"

0       True
1       True
2       True
3       True
4       True
        ... 
3528    True
3529    True
3530    True
3531    True
3532    True
Name: gp_business_status, Length: 3533, dtype: bool

In [14]:
# only keep restaurants that are operational 
gp = gp[gp["gp_business_status"] == "OPERATIONAL"].copy()

In [15]:
# helper function used normalize restaurant names
def normalize_name(s):
    s = s.fillna("").str.lower()
    s = s.str.replace(r"&", " and ", regex=True)
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

In [16]:
# create a new column to save normalized restaurant names
gp["gp_name_norm"] = normalize_name(gp["gp_name"].astype(str))

In [17]:
# Map Google Place price level to 1-4
gp_price_map = {"PRICE_LEVEL_INEXPENSIVE": 1,"PRICE_LEVEL_MODERATE": 2,"PRICE_LEVEL_EXPENSIVE": 3,"PRICE_LEVEL_VERY_EXPENSIVE": 4}
gp["gp_price_level_1to4"] = (gp["gp_price_level"].astype(str).str.strip().str.upper().map(gp_price_map))
gp["gp_price_level_1to4"] = gp["gp_price_level_1to4"].astype("Int64")

In [18]:
# extract info and save into new columns gp_price_start and gp_price_end from gp_price_range_raw
gp["gp_price_start"] = pd.to_numeric(gp["gp_price_range_raw"].astype(str).str.extract(r"'startPrice':\s*\{[^}]*'units':\s*'(\d+)'")[0], errors="coerce")
gp["gp_price_end"] = pd.to_numeric(gp["gp_price_range_raw"].astype(str).str.extract(r"'endPrice':\s*\{[^}]*'units':\s*'(\d+)'")[0], errors="coerce")

In [19]:
# helper function converting time to minutes
def convert_to_minutes(day, hour, minute):
    return int(day) * 24 * 60 + int(hour) * 60 + int(minute)

# extract each opening period (open.day, open.hour, open.minute, close.day, close.hour, close.minute)
PERIOD_PATTERN = re.compile(
    r"'open':\s*\{'day':\s*(\d+),\s*'hour':\s*(\d+),\s*'minute':\s*(\d+)\}\s*,\s*"
    r"'close':\s*\{'day':\s*(\d+),\s*'hour':\s*(\d+),\s*'minute':\s*(\d+)\}"
)

# parse gp_regular_opening_hours_raw using regex
def parse_opening_hours_summary_regex(x, late_threshold_hour=22):

    if pd.isna(x) or str(x).strip() == "":
        return pd.Series([pd.NA, pd.NA, pd.NA, pd.NA, pd.NA])

    text = str(x)

    matches = PERIOD_PATTERN.findall(text)
    if not matches:
        return pd.Series([0, 0, 0, 0, 0])

    minutes_by_day = [0] * 7 # e.g.: 0 = Sunday; 6 = Saturday
    total_minutes = 0
    open_late_minutes = 0
    late_cutoff = late_threshold_hour * 60

    for (open_day, open_hour, open_minute, close_day, close_hour, close_minute) in matches:

        start = convert_to_minutes(open_day, open_hour, open_minute)
        end = convert_to_minutes(close_day, close_hour, close_minute)

        if end <= start: # handle overnight time
            end += 7 * 24 * 60

        total_minutes += (end - start)

        # deal witth time across day boundaries; compute late minutes
        current_minute = start
        while current_minute < end:
            day_index = (current_minute // (24 * 60)) % 7
            day_start = (current_minute // (24 * 60)) * 24 * 60
            day_end = day_start + 24 * 60
            chunk_end = min(end, day_end)

            chunk_minutes = chunk_end - current_minute
            minutes_by_day[day_index] += chunk_minutes

            current_minute_of_day = current_minute - day_start
            chunk_end_minute_of_day = chunk_end - day_start
            late_start = max(current_minute_of_day, late_cutoff)
            late_end = max(min(chunk_end_minute_of_day, 24 * 60), late_start)
            open_late_minutes += max(0, late_end - late_start)

            current_minute = chunk_end

    open_days = sum(1 for m in minutes_by_day if m > 0)
    avg_open_minutes = round((total_minutes / open_days), 2) if open_days > 0 else 0
    weekend_minutes = minutes_by_day[0] + minutes_by_day[6]  # Sun + Sat

    return pd.Series([open_days, total_minutes, avg_open_minutes, weekend_minutes, open_late_minutes])

gp[["gp_open_days_per_week","gp_total_open_minutes_week","gp_avg_open_minutes_open_day","gp_weekend_open_minutes","gp_open_late_minutes"
    ]] = gp["gp_regular_opening_hours_raw"].apply(parse_opening_hours_summary_regex)

In [20]:
# clea up gp_address
ADDRESS_PATTERN = re.compile(r",\s*([^,]+)\s*,\s*([A-Z]{2})\s+(\d{5})\s*,\s*([^,]+)\s*$")

def extract_city_country_from_address(addr):
    if pd.isna(addr):
        return pd.Series([pd.NA, pd.NA])

    m = ADDRESS_PATTERN.search(str(addr))
    if not m:
        return pd.Series([pd.NA, pd.NA])

    city = m.group(1).strip()
    country = m.group(4).strip()
    return pd.Series([city, country])

# identify rows where city/country is missing or error
needs_fix = (gp["gp_city"].isna() | gp["gp_country"].isna()| gp["gp_city"].astype(str).str.contains(r"\d|,", regex=True, na=False))

gp.loc[needs_fix, ["gp_city", "gp_country"]] = gp.loc[needs_fix, "gp_address"].apply(extract_city_country_from_address)

In [21]:
# fix edge-case rows (city/country is missing or error)

# "street..., Chicago, IL 60617, USA"
PATTERN_WITH_STREET = re.compile(r",\s*([^,]+)\s*,\s*([A-Z]{2})\s+(\d{5})\s*,\s*([^,]+)\s*$")

# "Chicago, IL 60617, USA" (no street)
PATTERN_NO_STREET = re.compile(r"^\s*([^,]+)\s*,\s*([A-Z]{2})\s+(\d{5})\s*,\s*([^,]+)\s*$")

# get city and country from gp_address
def extract_city_country(addr):
    if pd.isna(addr):
        return None

    s = str(addr).strip()
    m = PATTERN_WITH_STREET.search(s)
    if m:
        return m.group(1).strip(), m.group(4).strip()

    m = PATTERN_NO_STREET.match(s)
    if m:
        return m.group(1).strip(), m.group(4).strip()
    
    return None

needs_fix = gp["gp_city"].isna() | gp["gp_country"].isna() # only patch rows where city/country is missing

patched = 0
for idx in gp[needs_fix].index:
    res = extract_city_country(gp.loc[idx, "gp_address"])
    if res is not None:
        gp.loc[idx, "gp_city"] = res[0]
        gp.loc[idx, "gp_country"] = res[1]
        patched += 1

print("missing gp_city:", gp["gp_city"].isna().sum())
print("missing gp_country:", gp["gp_country"].isna().sum())

missing gp_city: 0
missing gp_country: 0


In [22]:
# turn GP TRUE/FALSE columns into dummy columns; TRUE = 1; FALSE = 0
gp_cols = ["gp_takeout", "gp_delivery", "gp_dine_in", "gp_reservable"]

for c in gp_cols:
    gp[c + "_d"] = (gp[c].astype(str).str.strip().str.upper().map({"TRUE": 1, "FALSE": 0}).fillna(0) .astype(int))

gp[[c + "_d" for c in gp_cols]].head()

,gp_takeout_d,gp_delivery_d,gp_dine_in_d,gp_reservable_d
0,1,1,1,1
1,1,1,0,0
2,1,1,1,0
3,1,1,1,1
4,1,0,1,0


In [23]:
# Save cleaned gp_chicago as a new CSV
SAVE_GP_FILENAME = "Google_Place_Chicago_restaurants/gp_chicago_clean.csv"
SAVE_GP_FILEPATH = os.path.join(BASE_DIR, SAVE_GP_FILENAME)
gp.to_csv(SAVE_GP_FILEPATH, index=False)

## Clean up yelp_chicago

In [24]:
YELP_FILENAME = "Yelp_Chicago_restaurants/yelp_chicago.csv"
YELP_FILEPATH = os.path.join(BASE_DIR, YELP_FILENAME)
yelp = pd.read_csv(YELP_FILEPATH)

In [25]:
# Count unique yelp_business_id
unique_businesses = yelp["yelp_business_id"].nunique(dropna=True)
unique_businesses

6103

In [27]:
# Count non-missing yelp_price rows
non_missing_price_rows = yelp["yelp_price"].notna().sum()
non_missing_price_rows

np.int64(3709)

In [28]:
# proportion of non-missing yelp_price
non_missing_price_rows / len(yelp)

np.float64(0.6077339013599868)

In [29]:
# Unique businesses with non-missing yelp_price
yelp_dedup = yelp.drop_duplicates(subset=["yelp_business_id"])
non_missing_price_unique = yelp_dedup["yelp_price"].notna().sum()
non_missing_price_unique

np.int64(3709)

In [30]:
# Deduplicate
yelp = yelp.drop_duplicates(subset=["yelp_business_id"]).copy()

In [31]:
# Standardize types
yelp["yelp_latitude"] = pd.to_numeric(yelp["yelp_latitude"], errors="coerce")
yelp["yelp_longitude"] = pd.to_numeric(yelp["yelp_longitude"], errors="coerce")
yelp["yelp_rating"] = pd.to_numeric(yelp["yelp_rating"], errors="coerce")
yelp["yelp_review_count"] = pd.to_numeric(yelp["yelp_review_count"], errors="coerce").fillna(0).astype(int)

In [32]:
yelp["yelp_name_norm"] = normalize_name(yelp["yelp_name"].astype(str))

In [33]:
yelp["yelp_zip_code"] = to_zip_str(yelp["yelp_zip_code"].astype(str))

In [34]:
# Map Yelp price level $-$$$$ to 1-4
price_map = {"$": 1, "$$": 2, "$$$": 3, "$$$$": 4}
yelp["yelp_price_level"] = yelp["yelp_price"].map(price_map)
yelp["yelp_price_level"] = yelp["yelp_price_level"].astype("Int64")

In [35]:
# Filter Yelp to Chicago zipcode
chicago_zip = set(gp["gp_zipcode"].dropna().unique())
yelp = yelp[(yelp["yelp_state"] == "IL") & (yelp["yelp_zip_code"].isin(chicago_zip))].copy()
yelp["yelp_price_level_1to4"] = yelp["yelp_price_level"]
yelp = yelp.drop(columns=["yelp_price_level"])

In [36]:
# Create a new column to save normalized restaurant names
yelp["yelp_name_norm"] = normalize_name(yelp["yelp_name"].astype(str))

In [37]:
# Split yelp_transactions into dummy columns (delivery,pickup,restaurant_reservation)
tx = (yelp["yelp_transactions"].fillna("").astype(str).str.lower().str.replace(" ", "", regex=False))

yelp["yelp_delivery_d"] = tx.str.contains(r"(^|,)delivery(,|$)", regex=True).astype(int)
yelp["yelp_pickup_d"] = tx.str.contains(r"(^|,)pickup(,|$)", regex=True).astype(int)
yelp["yelp_restaurant_reservation_d"] = tx.str.contains(r"(^|,)restaurant_reservation(,|$)", regex=True).astype(int)

yelp[["yelp_delivery_d", "yelp_pickup_d", "yelp_restaurant_reservation_d"]].sum()

/var/folders/lm/wvqh87x93hs7jfg66h8zq72c0000gn/T/ipykernel_84737/3326919430.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  yelp["yelp_delivery_d"] = tx.str.contains(r"(^|,)delivery(,|$)", regex=True).astype(int)
/var/folders/lm/wvqh87x93hs7jfg66h8zq72c0000gn/T/ipykernel_84737/3326919430.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  yelp["yelp_pickup_d"] = tx.str.contains(r"(^|,)pickup(,|$)", regex=True).astype(int)
/var/folders/lm/wvqh87x93hs7jfg66h8zq72c0000gn/T/ipykernel_84737/3326919430.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  yelp["yelp_restaurant_reservation_d"] = tx.str.contains(r"(^|,)restaurant_reservation(,|$)", regex=True).astype(int)


yelp_delivery_d                  3601
yelp_pickup_d                    3280
yelp_restaurant_reservation_d      61
dtype: int64

In [ ]:
# Save cleaned gp_chicago as a new CSV
SAVE_YELP_FILENAME = "Yelp_Chicago_restaurants/yelp_chicago_clean.csv"
SAVE_YELP_FILEPATH = os.path.join(BASE_DIR, SAVE_YELP_FILENAME)
yelp.to_csv(SAVE_YELP_FILEPATH, index=False)